In [4]:
import argparse
import datetime
import math
import os
from time import time
from typing import Any, List, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from analytical_return import (
    compute_objective_via_analytical,
)
from artifacts import (
    build_plot_df_wrapper,
    save_csv_artifact,
    save_plot_strategy,
)
from data import (
    apply_final_treatment,
    join_metadata,
    load_metadata_artefacts,
    load_odds,
)
from dependencies.config import load_config
from dependencies.utils import get_bet_return, softmax
from filter import filter_by_linear_combination
from GameProbs import GameProbs
from Optimizer import Optimizer

config = load_config("config/config.yml")
metadata, gameid_to_outcome = load_metadata_artefacts(config.metadata_path)
odds = load_odds(config.odds_path, n=5)
odds = join_metadata(odds, metadata)

odds = odds.sort_values(["Datetime", "GameId"], ascending=True)

#odds = odds[(odds.Datetime.apply(str)>"2021-01-01")&(odds.Datetime.apply(str)<="2021-02-01")]
odds = odds[(odds.Datetime.apply(str)=="2019-06-01")]

def process_group(group: Tuple[str, pd.DataFrame], args) -> List[List[Any]]:
    
    is_valid_solution = True

    _, group_data = group
   
    games_ids = group_data['GameId'].unique()
    
    # Initialize dict to store dataframes of favorable bet opportunities
    odds_dict = {}
    # Initialize dict to store 7x7 matrices/dataframes of real probabilities 
    df_probs_dict = {}

    if len(games_ids) > args.min_games:

        for game_id in games_ids:
            df = GameProbs(game_id).build_dataframe()
            
            odds_sample = group_data[(group_data.GameId==game_id)]
            odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
            if not args.do_baseline:   
                odds_sample = filter_by_linear_combination(odds_sample)
            else:
                odds_sample = odds_sample.sample(1)
            odds_dict[game_id] = odds_sample
            df_probs_dict[game_id] = df

        odds_dt = pd.concat(odds_dict.values())

        if len(odds_dt) <= config.max_vector_length:
            iteration_date = odds_dt.Datetime.apply(str).unique()[0]
            print(f"Date: {iteration_date}")

            odds_favorable = np.array(odds_dt['Odd'])
            real_prob_favorable = np.array(odds_dt['real_prob'])
            event_favorable = list(odds_dt['BetMap'].values)
            games_ids = np.array(odds_dt['GameId'])

            #try:
            print("Execution of minimization task...")    
            solution, time_limit_flag = Optimizer().run_optimization(
                fun=compute_objective_via_analytical,
                x0=np.zeros(len(odds_favorable)),
                args=(odds_favorable, real_prob_favorable, event_favorable, games_ids, df_probs_dict)
            )               
            print("Finalization of minimization task...")

            #except ValueError:
                #continue
            
            if any(math.isnan(x) for x in solution):
                is_valid_solution = False

            print(f"sum of solution: {sum(solution)}")
            print(f"solution: {np.round(solution, 3)}")
            odds_dt['solution'] = softmax(solution)

            track_record = []

            for game_id, game_data in odds_dt.groupby('GameId', sort=False):
                scenario = gameid_to_outcome[game_id]
                financial_return = get_bet_return(df=game_data,
                                                  allocation_array=game_data.solution,
                                                  scenario=scenario)

                print(f"game_id: {game_id}; financial_return: {financial_return}")

                track_record.append([str(game_id),
                                     financial_return,
                                     len(game_data),
                                     time_limit_flag,
                                     is_valid_solution,
                                     iteration_date])

            return track_record

In [5]:
def run_strategy(args):
    
    start_time = time()

    grouped = odds.groupby(args.aggregator)
    
    # Use all available CPU cores for parallel execution
    num_jobs = 1
    # Parallelize the group processing
    results = Parallel(n_jobs=num_jobs)(delayed(process_group)(group, args) for group in grouped)

    data = [x for x in results if x is not None]
    df_flat = pd.DataFrame([item for sublist in data for item in sublist])

    if args.save_experiment:

        # Create artefacts folder
        timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
        artefacts_folder = f"artefacts/aggregator{args.aggregator}_min_games{args.min_games}_do_baseline{args.do_baseline}_{timestamp}"
        os.makedirs(artefacts_folder)
        args.artefacts_folder = artefacts_folder

        save_csv_artifact(artefacts_folder, "result", df_flat)
        df_plot = build_plot_df_wrapper(args)
        save_csv_artifact(artefacts_folder, "result_plot", df_plot)
        save_plot_strategy(args, df_plot)
    
    elapsed_time = time() - start_time
    print("Final Elapsed: %.3f sec" % elapsed_time)

In [6]:
#if __name__ == "__main__":
parser = argparse.ArgumentParser()
parser.add_argument(
    "--aggregator",
    type=str,
    help="aggregate by GameId or by Datetime"
)
parser.add_argument(
    "--min_games",
    type=int,
    default=0,
    help="threshold of minimum number of games to enter the optimization task"
)
parser.add_argument(
    "--do_baseline",
    action='store_true',
    help="flag to apply baseline logic or not, not specifying the argument return the opposite of the action"
)
parser.add_argument(
    "--save_experiment",
    action='store_true',
    help="flag to save the experiment artefacts, not specifying the argument return the opposite of the action"
)
#args = parser.parse_args()
args = parser.parse_args([
    "--aggregator", "Datetime",
    "--min_games", "1",
    #"--do_baseline", "False"
    #"--save_experiment"
])
print(args)
run_strategy(args)


Namespace(aggregator='Datetime', min_games=1, do_baseline=False, save_experiment=False)
Date: 2019-06-01
Execution of minimization task...
Optimization terminated successfully    (Exit mode 0)
            Current function value: -1.603408731597838
            Iterations: 10
            Function evaluations: 100
            Gradient evaluations: 10
Finalization of minimization task...
sum of solution: 6.983537668574792
solution: [ 5.984  4.783 -6.251  3.748 -6.724  4.947  3.049  1.668 -4.22 ]
game_id: 2782628; financial_return: 1.396671486647671
game_id: 2782629; financial_return: 0.0
Final Elapsed: 0.286 sec
